# Stock History Loader

This notebook pulls stock history into the local AlphaLens bronze/silver/gold stores.
It uses the notebook-oriented loader in `fg.pipelines.historical_loader`, including the SEC canonicalization pass needed for live `companyfacts` payloads.

In [1]:
from fg.pipelines import HistoricalDataLoader
from fg.settings import get_settings

settings = get_settings()
loader = HistoricalDataLoader(settings)
settings.data_dirs

{'root': WindowsPath('C:/Users/David/Desktop/AlphaLens/data'),
 'duckdb_path': WindowsPath('C:/Users/David/Desktop/AlphaLens/data/fg.duckdb'),
 'bronze': WindowsPath('C:/Users/David/Desktop/AlphaLens/data/bronze'),
 'silver': WindowsPath('C:/Users/David/Desktop/AlphaLens/data/silver'),
 'gold': WindowsPath('C:/Users/David/Desktop/AlphaLens/data/gold'),
 'cache': WindowsPath('C:/Users/David/Desktop/AlphaLens/data/cache'),
 'exports': WindowsPath('C:/Users/David/Desktop/AlphaLens/data/exports'),
 'logs': WindowsPath('C:/Users/David/Desktop/AlphaLens/data/logs')}

In [2]:
TICKERS = ["AAPL", "MSFT", "KO"]
WATCHLIST_NAME = None
USE_FIXTURES = False
BUILD_GOLD = True
LOOKBACK_YEARS = 20
PE_METHOD = "static_15"
SHOW_ESTIMATES = True
MANUAL_PE = None
CONTINUE_ON_ERROR = True

In [ ]:
if WATCHLIST_NAME:
    TICKERS = [
        str(ticker).upper()
        for ticker in settings.watchlists_config.get("watchlists", {}).get(WATCHLIST_NAME, [])
    ]

summary_df = loader.load_tickers(
    TICKERS,
    fixture_mode=USE_FIXTURES,
    build_gold=BUILD_GOLD,
    lookback_years=LOOKBACK_YEARS,
    pe_method=PE_METHOD,
    show_estimates=SHOW_ESTIMATES,
    manual_pe=MANUAL_PE,
    continue_on_error=CONTINUE_ON_ERROR,
)
summary_df

,ticker,company_key,status,issuer_name,warning_count,bronze_sec_submissions,bronze_sec_companyfacts,bronze_yahoo_prices,bronze_yahoo_actions,bronze_fmp_estimates,...,silver_price_daily,silver_price_monthly,silver_corporate_action,silver_estimate_snapshot,silver_quality_issue,gold_valuation_series,gold_eps_bars,gold_kpi_snapshot,gold_source_freshness,gold_audit_grid
0,AAPL,0000320193,ok,Apple Inc.,1,1,1,11399,95,1,...,11399,544,95,0,1,669,24,1,3,320
1,MSFT,0000789019,ok,MICROSOFT CORP,1,1,1,10073,98,1,...,10073,481,98,0,1,606,24,1,3,319
2,KO,0000021344,ok,COCA COLA CO,1,1,1,16152,264,1,...,16152,771,264,0,1,896,24,1,3,271


In [4]:
loader.table_inventory()

,layer,table_name,file_count,row_count
0,bronze,bronze_fmp_estimates,6,9
1,bronze,bronze_sec_companyfacts,6,407
2,bronze,bronze_sec_submissions,6,407
3,bronze,bronze_yahoo_actions,6,481
4,bronze,bronze_yahoo_prices,6,38515
5,gold,mart_audit_grid,6,883
6,gold,mart_eps_bars,6,72
7,gold,mart_kpi_snapshot,6,12
8,gold,mart_source_freshness,6,36
9,gold,mart_valuation_series,7,2290


In [5]:
created_views = loader.register_duckdb_views()
created_views

['v_sec_submissions',
 'v_sec_companyfacts',
 'v_yahoo_prices',
 'v_yahoo_actions',
 'v_fmp_estimates',
 'v_dim_company',
 'v_fundamental_annual',
 'v_fundamental_quarterly',
 'v_fundamental_ttm',
 'v_price_daily',
 'v_price_monthly',
 'v_corporate_action',
 'v_estimate_snapshot',
 'v_quality_issue',
 'v_valuation_series',
 'v_eps_bars',
 'v_kpi_snapshot',
 'v_source_freshness',
 'v_audit_grid']

In [6]:
loader.query_duckdb(
    """
    SELECT ticker, company_key, issuer_name, last_sec_pull_at, last_yahoo_pull_at, last_fmp_pull_at
    FROM v_dim_company
    ORDER BY ticker
    """
)

,ticker,company_key,issuer_name,last_sec_pull_at,last_yahoo_pull_at,last_fmp_pull_at
0,AAPL,320193,Apple Inc.,2026-03-07,2026-03-07,2026-03-07
1,AAPL,0000320193,Apple Inc.,2026-03-07,2026-03-07,2026-03-07
2,KO,21344,COCA COLA CO,2026-03-07,2026-03-07,2026-03-07
3,KO,0000021344,COCA COLA CO,2026-03-07,2026-03-07,2026-03-07
4,MSFT,789019,MICROSOFT CORP,2026-03-07,2026-03-07,2026-03-07
5,MSFT,0000789019,MICROSOFT CORP,2026-03-07,2026-03-07,2026-03-07


In [7]:
loader.query_duckdb(
    """
    SELECT company_key, metric_code, COUNT(*) AS row_count
    FROM v_fundamental_annual
    GROUP BY company_key, metric_code
    ORDER BY company_key, metric_code
    """
)

,company_key,metric_code,row_count
0,0000021344,eps_diluted_actual,17
1,0000021344,net_income_actual,17
2,0000021344,revenue_actual,8
3,0000021344,shares_diluted_actual,17
4,0000320193,eps_diluted_actual,17
5,0000320193,net_income_actual,17
6,0000320193,revenue_actual,17
7,0000320193,shares_diluted_actual,17
8,0000789019,eps_diluted_actual,16
9,0000789019,net_income_actual,16
